# Module 36: Custom C++ Extensions — When Python Isn't Fast Enough

**Prerequisites**: Module 04 (Neural Networks), Module 35 (The Dispatcher)  
**Time**: ~3 hours  

PyTorch is fast because it calls optimized C++/CUDA kernels under the hood. But what if you need a kernel that doesn't exist? This module teaches you to write your own.

---

## 1. Why C++ Extensions?

Three situations where Python isn't enough:

1. **Performance-critical custom ops** — tight inner loops where Python overhead kills throughput
2. **Custom CUDA kernels** — novel GPU algorithms not covered by existing PyTorch ops
3. **Wrapping C/C++ libraries** — integrating existing numerical code with PyTorch

PyTorch's `torch.utils.cpp_extension` module handles all the build complexity: compiler invocation, include paths, ABI compatibility, and caching.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.cpp_extension import CppExtension, CUDAExtension, BuildExtension

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")

## 2. Anatomy of a C++ Extension

A minimal C++ extension has three key ingredients:
1. `#include <torch/extension.h>` — the single header for PyTorch C++ API + pybind11
2. Your C++ functions using `torch::Tensor`
3. `PYBIND11_MODULE` — creates the Python module

In [ ]:
# The C++ source code for a minimal extension
MINIMAL_CPP = r"""
// my_add.cpp
#include <torch/extension.h>

torch::Tensor my_add(torch::Tensor a, torch::Tensor b) {
    TORCH_CHECK(a.sizes() == b.sizes(), "Size mismatch");
    return a + b;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("my_add", &my_add, "Element-wise addition");
}
"""

print("=== my_add.cpp ===")
print(MINIMAL_CPP)
print("Key components:")
print("  torch::Tensor        — C++ equivalent of torch.Tensor")
print("  TORCH_CHECK          — Throws RuntimeError on failure")
print("  PYBIND11_MODULE      — Creates Python bindings")
print("  TORCH_EXTENSION_NAME — Auto-set from load()/CppExtension()")

## 3. The `load()` JIT Compilation API

The easiest way to build a C++ extension — compiles on first call, caches the `.so`.

In [ ]:
# How you'd use load() in practice
print("""
from torch.utils.cpp_extension import load

# Basic usage
module = load(
    name="my_add",              # Module name
    sources=["my_add.cpp"],     # Source files to compile
    verbose=True,               # Print compile commands
)

# Use it like any Python function
result = module.my_add(torch.ones(3), torch.ones(3))
# tensor([2., 2., 2.])
""")

# What happens under the hood:
print("Compilation process:")
print("  1. Check ~/.cache/torch_extensions/ for cached .so")
print("  2. If not cached: invoke g++ with PyTorch headers")
print("  3. Link into shared library (.so)")
print("  4. importlib.import_module() to load it")
print("  5. Return module with bound functions")

In [ ]:
# Advanced load() with extra flags
print("""
module = load(
    name="fast_ops",
    sources=["ops.cpp", "kernels.cu"],
    extra_include_paths=["/usr/local/include"],
    extra_cflags=["-O3", "-march=native", "-fopenmp"],
    extra_cuda_cflags=["-O3", "--use_fast_math"],
    extra_ldflags=["-lgomp"],
    verbose=True,
)
""")

# load_inline() — skip writing files entirely
print("""
from torch.utils.cpp_extension import load_inline

cpp_source = '''
torch::Tensor double_it(torch::Tensor x) { return x * 2; }
'''

module = load_inline(
    name="inline_ext",
    cpp_sources=cpp_source,
    functions=["double_it"],
)
""")

# Show PyTorch's include paths
from torch.utils.cpp_extension import include_paths
print("PyTorch include paths (added automatically):")
for p in include_paths():
    print(f"  {p}")

## 4. `setup.py` for Distribution

For distributable packages, use `CppExtension` / `CUDAExtension` with setuptools.

In [ ]:
# CPU-only extension
SETUP_CPU = """
from setuptools import setup
from torch.utils.cpp_extension import CppExtension, BuildExtension

setup(
    name="my_cpu_ext",
    ext_modules=[
        CppExtension(
            name="my_cpu_ext",
            sources=["my_ext.cpp"],
            extra_compile_args=["-O3"],
        ),
    ],
    cmdclass={"build_ext": BuildExtension},
)
"""

# CUDA extension
SETUP_CUDA = """
from setuptools import setup
from torch.utils.cpp_extension import CUDAExtension, BuildExtension

setup(
    name="my_cuda_ext",
    ext_modules=[
        CUDAExtension(
            name="my_cuda_ext",
            sources=["ops.cpp", "kernels.cu"],
            extra_compile_args={
                "cxx": ["-O3"],
                "nvcc": ["-O3", "--use_fast_math"],
            },
        ),
    ],
    cmdclass={"build_ext": BuildExtension},
)
"""

print("=== CPU setup.py ===")
print(SETUP_CPU)
print("=== CUDA setup.py ===")
print(SETUP_CUDA)
print("Install: pip install .")
print("Wheel:   python setup.py bdist_wheel")

## 5. Writing CUDA Kernels

CUDA extensions have two parts:
- `.cu` file — GPU kernels compiled by nvcc
- `.cpp` file — Python bindings compiled by gcc

In [ ]:
# Complete CUDA kernel: fused add + ReLU
CUDA_KERNEL = r"""
// fused_add_relu.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

template <typename scalar_t>
__global__ void fused_add_relu_kernel(
    const scalar_t* __restrict__ a,
    const scalar_t* __restrict__ b,
    scalar_t* __restrict__ output,
    int64_t size
) {
    const int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        scalar_t val = a[idx] + b[idx];
        output[idx] = val > 0 ? val : 0;  // ReLU
    }
}

torch::Tensor fused_add_relu_cuda(torch::Tensor a, torch::Tensor b) {
    auto output = torch::empty_like(a);
    const int64_t size = a.numel();
    const int threads = 256;
    const int blocks = (size + threads - 1) / threads;

    AT_DISPATCH_FLOATING_TYPES(a.scalar_type(), "fused_add_relu", [&] {
        fused_add_relu_kernel<scalar_t><<<blocks, threads>>>(
            a.data_ptr<scalar_t>(),
            b.data_ptr<scalar_t>(),
            output.data_ptr<scalar_t>(),
            size
        );
    });
    return output;
}
"""

print(CUDA_KERNEL)

In [ ]:
# Explain the CUDA concepts
print("CUDA kernel concepts:")
print()
print("__global__          — Function runs on GPU, called from CPU")
print("__restrict__        — Pointers don't alias (enables optimizations)")
print("<<<blocks, threads>>> — Kernel launch configuration")
print()
print("Thread indexing:")
print("  blockIdx.x  = which block (0, 1, 2, ...)")
print("  blockDim.x  = threads per block (256)")
print("  threadIdx.x = position within block (0-255)")
print("  global_idx  = blockIdx.x * blockDim.x + threadIdx.x")
print()
print("Grid/block sizing:")
for n in [1000, 100_000, 10_000_000]:
    threads = 256
    blocks = (n + threads - 1) // threads
    print(f"  {n:>12,} elements → {blocks:>6,} blocks × {threads} threads")

## 6. Accessing Tensor Data in C++

Four patterns for working with tensor data, from fastest (unsafe) to safest.

In [ ]:
print("""
Pattern 1: Raw pointer (fastest, no bounds checking)
    auto t = tensor.contiguous();  // MUST ensure contiguity!
    float* data = t.data_ptr<float>();

Pattern 2: Accessor (safe, handles strides)
    auto acc = tensor.accessor<float, 2>();
    acc[i][j] = 1.0f;  // Bounds-checked in debug mode

Pattern 3: Packed accessor for CUDA
    auto pacc = tensor.packed_accessor32<float, 2, RestrictPtrTraits>();
    // Pass to kernel — 32-bit indexing for speed

Pattern 4: AT_DISPATCH for dtype-generic code
    AT_DISPATCH_FLOATING_TYPES(tensor.scalar_type(), "op", [&] {
        // scalar_t is float or double
        auto data = tensor.data_ptr<scalar_t>();
    });
""")

# Python equivalents
t = torch.randn(3, 4)
print(f"Shape:          {t.shape}")
print(f"Strides:        {t.stride()}")
print(f"Contiguous:     {t.is_contiguous()}")
print(f"Data pointer:   {t.data_ptr()} (memory address)")
print(f"Dtype:          {t.dtype}")
print(f"Device:         {t.device}")

## 7. Autograd Integration in C++

To support `backward()`, write a custom autograd function in C++.

In [ ]:
AUTOGRAD_CPP = r"""
#include <torch/extension.h>
using namespace torch::autograd;

class FusedLinearReLU : public Function<FusedLinearReLU> {
public:
    static torch::Tensor forward(
        AutogradContext* ctx,
        torch::Tensor input, torch::Tensor weight, torch::Tensor bias
    ) {
        auto output = torch::relu(torch::mm(input, weight.t()) + bias);
        ctx->save_for_backward({input, weight, output});
        return output;
    }

    static tensor_list backward(
        AutogradContext* ctx, tensor_list grad_outputs
    ) {
        auto saved = ctx->get_saved_variables();
        auto input = saved[0], weight = saved[1], output = saved[2];
        auto grad = grad_outputs[0];

        auto grad_relu = grad * (output > 0).to(grad.dtype());
        return {
            torch::mm(grad_relu, weight),       // grad_input
            torch::mm(grad_relu.t(), input),    // grad_weight
            grad_relu.sum(0)                    // grad_bias
        };
    }
};
"""

print(AUTOGRAD_CPP)

In [ ]:
# Python equivalent — verify the autograd logic
class FusedLinearReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight, bias):
        output = F.relu(input @ weight.t() + bias)
        ctx.save_for_backward(input, weight, output)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, weight, output = ctx.saved_tensors
        grad_relu = grad_output * (output > 0).float()
        return grad_relu @ weight, grad_relu.t() @ input, grad_relu.sum(0)

# Test
x = torch.randn(4, 8, requires_grad=True)
w = torch.randn(16, 8, requires_grad=True)
b = torch.randn(16, requires_grad=True)

out = FusedLinearReLU.apply(x, w, b)
out.sum().backward()

print(f"Output shape: {out.shape}")
print(f"x.grad shape: {x.grad.shape}")
print(f"w.grad shape: {w.grad.shape}")
print(f"b.grad shape: {b.grad.shape}")

# Gradient check
x64 = torch.randn(4, 8, dtype=torch.float64, requires_grad=True)
w64 = torch.randn(16, 8, dtype=torch.float64, requires_grad=True)
b64 = torch.randn(16, dtype=torch.float64, requires_grad=True)
print(f"Gradient check: {torch.autograd.gradcheck(FusedLinearReLU.apply, (x64, w64, b64))}")

## 8. Error Handling: `TORCH_CHECK`

`TORCH_CHECK` is the standard way to validate inputs in C++ extensions. It throws a `c10::Error` that surfaces as a Python `RuntimeError`.

In [ ]:
print("""
=== Error handling patterns in C++ extensions ===

// User-facing validation (becomes RuntimeError in Python)
TORCH_CHECK(input.dim() == 2,
    "Expected 2D input, got ", input.dim(), "D");

TORCH_CHECK(input.device().is_cuda(),
    "Input must be on CUDA, got ", input.device());

TORCH_CHECK(input.scalar_type() == torch::kFloat32,
    "Expected float32, got ", input.scalar_type());

TORCH_CHECK(weight.size(1) == input.size(1),
    "Weight cols (", weight.size(1),
    ") must match input cols (", input.size(1), ")");

// Internal invariants (can be compiled away in release)
TORCH_INTERNAL_ASSERT(output.numel() == input.numel());

// CUDA error checking
cudaError_t err = cudaMemcpy(dst, src, n, cudaMemcpyDeviceToDevice);
TORCH_CHECK(err == cudaSuccess,
    "CUDA error: ", cudaGetErrorString(err));
""")

# Python-side: these become RuntimeError
print("Best practices:")
print("  - Always include actual values in error messages")
print("  - Check device, dtype, shape at the function entry point")
print("  - Use TORCH_CHECK for user errors, TORCH_INTERNAL_ASSERT for bugs")

## 9. Performance Tips

Key optimizations for C++ and CUDA extensions.

In [ ]:
print("""
=== CPU Performance ===

1. Ensure contiguity before raw pointer access:
   auto t = tensor.contiguous();
   float* ptr = t.data_ptr<float>();

2. Use OpenMP for CPU parallelism:
   #pragma omp parallel for
   for (int64_t i = 0; i < n; i++) { ... }

3. Use NoGradGuard for inference:
   torch::NoGradGuard no_grad;

=== CUDA Performance ===

4. Memory coalescing — adjacent threads access adjacent memory:
   output[idx] = input[idx] * 2;  // Good: coalesced
   output[idx] = input[idx * stride];  // Bad: strided

5. Minimize kernel launches — fuse operations:
   // Bad: 3 launches
   auto t1 = a + b;
   auto t2 = t1 * c;
   auto out = torch::relu(t2);
   // Good: 1 launch (custom fused kernel)

6. Use shared memory for data reuse (tiled algorithms)
7. Avoid branch divergence within warps
""")

In [ ]:
# Demonstrate fusion benefit (CPU simulation)
import time

x = torch.randn(10_000_000)
y = torch.randn(10_000_000)

# Unfused: 3 separate ops = 3 memory passes
start = time.perf_counter()
for _ in range(10):
    temp = x + y
    temp2 = temp * 2.0
    result = torch.relu(temp2)
unfused = (time.perf_counter() - start) / 10

# Single expression (PyTorch may optimize internally)
start = time.perf_counter()
for _ in range(10):
    result = torch.relu((x + y) * 2.0)
fused = (time.perf_counter() - start) / 10

print(f"3 separate ops:    {unfused*1000:.2f} ms")
print(f"Single expression: {fused*1000:.2f} ms")
print(f"A true fused CUDA kernel would do this in ONE memory pass")

## 10. CUDA Kernel Patterns

In [ ]:
# Pattern: Reduction using shared memory
REDUCTION = r"""
template <typename scalar_t>
__global__ void sum_reduction(
    const scalar_t* input, scalar_t* output, int64_t size
) {
    extern __shared__ char smem[];
    scalar_t* sdata = reinterpret_cast<scalar_t*>(smem);

    int tid = threadIdx.x;
    int gid = blockIdx.x * blockDim.x + threadIdx.x;

    sdata[tid] = (gid < size) ? input[gid] : 0;
    __syncthreads();

    // Tree reduction
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }

    if (tid == 0) atomicAdd(&output[0], sdata[0]);
}
"""
print("=== Reduction Kernel ===")
print(REDUCTION)
print("Key: __shared__ memory is fast on-chip storage shared within a block")
print("     __syncthreads() is a block-wide barrier")
print("     Tree reduction: O(log n) steps within each block")

In [ ]:
# Pattern: Fused dropout + scale
DROPOUT_KERNEL = r"""
template <typename scalar_t>
__global__ void fused_dropout_scale_kernel(
    const scalar_t* input,
    scalar_t* output,
    uint8_t* mask,
    int64_t size,
    float dropout_prob,
    float scale,           // 1/(1-p)
    unsigned long long seed
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= size) return;

    curandStatePhilox4_32_10_t state;
    curand_init(seed, idx, 0, &state);
    float rand = curand_uniform(&state);

    if (rand < dropout_prob) {
        output[idx] = 0;
        mask[idx] = 0;
    } else {
        output[idx] = input[idx] * scale;
        mask[idx] = 1;
    }
}
"""
print("=== Fused Dropout + Scale Kernel ===")
print(DROPOUT_KERNEL)
print("Saves one full memory read+write pass vs separate dropout and scale ops")

## 11. Modern Alternative: Triton

For many GPU kernels, [Triton](../25_triton_kernels/) (Python!) is simpler than CUDA C++.

In [ ]:
print("""
CUDA C++ vs Triton comparison:

  CUDA C++:                          Triton:
  - C++/CUDA language                - Python
  - Manual shared memory             - Automatic tiling
  - Manual memory coalescing         - Automatic
  - Manual grid search               - @triton.autotune
  - Needs dispatcher registration    - Native torch.compile support
  - Full hardware control            - ~90-95% of peak
  - Steep learning curve             - Moderate learning curve

Use CUDA C++ when:
  - You need warp-level primitives
  - Complex shared memory patterns
  - Wrapping existing CUDA libraries
  - Hardware intrinsics (TMA, async copies)

Use Triton when:
  - Elementwise, reduction, or attention kernels
  - You want autotuning
  - Development speed matters
  - You need torch.compile compatibility
""")

In [ ]:
# The triton_op approach — register Triton kernels as native ops
TRITON_OP = """
import torch, triton, triton.language as tl

@triton.jit
def add_relu_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, tl.maximum(x + y, 0.0), mask=mask)

@torch.library.custom_op("myops::add_relu", mutates_args=())
def add_relu(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(x)
    n = x.numel()
    add_relu_kernel[(triton.cdiv(n, 1024),)](x, y, out, n, BLOCK=1024)
    return out

@add_relu.register_fake
def _(x, y):
    return torch.empty_like(x)
"""

print(TRITON_OP)
print("This kernel:")
print("  - Integrates with the dispatcher")
print("  - Works with torch.compile")
print("  - Can be called as torch.ops.myops.add_relu()")
print("  - Is ~20 lines of Python vs ~50 lines of CUDA C++")

## 12. Dispatcher Registration (Modern Approach)

For full integration with `torch.compile`, register ops via `torch.library`.

In [ ]:
print("""
# Register a custom op with the dispatcher
import torch

# Step 1: Define the schema
torch.library.define(
    "myops::fused_linear_relu",
    "(Tensor input, Tensor weight, Tensor bias) -> Tensor"
)

# Step 2: Register CPU implementation
@torch.library.impl("myops::fused_linear_relu", "cpu")
def fused_lr_cpu(input, weight, bias):
    return torch.relu(input @ weight.t() + bias)

# Step 3: Register fake tensor impl (for torch.compile tracing)
@torch.library.register_fake("myops::fused_linear_relu")
def fused_lr_fake(input, weight, bias):
    return input.new_empty(input.shape[0], weight.shape[0])

# Step 4: Use it
result = torch.ops.myops.fused_linear_relu(x, w, b)

# Works with torch.compile!
@torch.compile
def compiled_fn(x, w, b):
    return torch.ops.myops.fused_linear_relu(x, w, b)
""")

## 13. Exercise: Fused Dropout + Scale Kernel

**Task**: Write the C++ source for a fused dropout+scale kernel. Explain each line.

The key insight: standard dropout requires two memory passes (generate mask, then apply mask+scale). A fused kernel does it in one pass.

In [ ]:
# Exercise solution: annotated fused dropout+scale

EXERCISE_SOLUTION = r"""
// fused_dropout_scale.cu
#include <torch/extension.h>        // PyTorch C++ API + pybind11
#include <cuda.h>                    // CUDA driver API
#include <cuda_runtime.h>            // CUDA runtime API
#include <curand_kernel.h>           // GPU random number generation

// CUDA kernel: each thread handles one element
template <typename scalar_t>          // Templated for float/double
__global__ void fused_dropout_scale_kernel(
    const scalar_t* __restrict__ input,   // Input tensor (read-only)
    scalar_t* __restrict__ output,        // Output tensor (write)
    uint8_t* __restrict__ mask,           // Binary mask (for backward)
    int64_t size,                         // Total number of elements
    float dropout_prob,                   // Probability of dropping
    float scale,                          // = 1.0 / (1.0 - dropout_prob)
    unsigned long long seed               // Random seed
) {
    // Compute this thread's global index
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= size) return;              // Guard for last block

    // Initialize per-thread random number generator
    // Philox is fast and produces high-quality randomness
    curandStatePhilox4_32_10_t state;
    curand_init(seed, idx, 0, &state);   // seed + unique offset per thread

    // Generate uniform random in [0, 1)
    float rand_val = curand_uniform(&state);

    // Apply dropout: zero out with probability p, scale rest by 1/(1-p)
    if (rand_val < dropout_prob) {
        output[idx] = 0;                 // Dropped
        mask[idx] = 0;                   // Record in mask
    } else {
        output[idx] = input[idx] * static_cast<scalar_t>(scale);  // Kept + scaled
        mask[idx] = 1;
    }
}

// Host function: validates inputs and launches kernel
std::tuple<torch::Tensor, torch::Tensor> fused_dropout_scale(
    torch::Tensor input,
    double dropout_prob,
    bool training
) {
    TORCH_CHECK(input.is_cuda(), "Input must be CUDA tensor");
    TORCH_CHECK(dropout_prob >= 0 && dropout_prob < 1,
        "dropout_prob must be in [0, 1), got ", dropout_prob);

    if (!training || dropout_prob == 0.0) {
        auto mask = torch::ones(input.sizes(),
            input.options().dtype(torch::kUInt8));
        return {input, mask};             // No-op during eval
    }

    auto output = torch::empty_like(input);
    auto mask = torch::empty(input.sizes(),
        input.options().dtype(torch::kUInt8));
    float scale = 1.0f / (1.0f - static_cast<float>(dropout_prob));

    const int threads = 256;
    const int blocks = (input.numel() + threads - 1) / threads;

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(
        input.scalar_type(), "fused_dropout_scale", [&] {
            fused_dropout_scale_kernel<scalar_t><<<blocks, threads>>>(
                input.data_ptr<scalar_t>(),
                output.data_ptr<scalar_t>(),
                mask.data_ptr<uint8_t>(),
                input.numel(),
                static_cast<float>(dropout_prob),
                scale,
                42ULL  // Fixed seed for reproducibility
            );
        }
    );

    return {output, mask};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("fused_dropout_scale", &fused_dropout_scale);
}
"""

print(EXERCISE_SOLUTION)

In [ ]:
# Verify with Python equivalent
def fused_dropout_scale_python(input, dropout_prob, training):
    """Python equivalent of the CUDA kernel above."""
    if not training or dropout_prob == 0.0:
        return input, torch.ones_like(input, dtype=torch.uint8)
    mask = (torch.rand_like(input) >= dropout_prob).to(torch.uint8)
    scale = 1.0 / (1.0 - dropout_prob)
    output = input * mask.float() * scale
    return output, mask

x = torch.randn(1000)
out, mask = fused_dropout_scale_python(x, 0.5, training=True)
print(f"Input:  {x[:10]}")
print(f"Output: {out[:10]}")
print(f"Mask:   {mask[:10]}")
print(f"Fraction kept: {mask.float().mean():.3f} (expect ~0.5)")
print(f"Scale applied: {(out[mask==1] / x[mask==1]).mean():.3f} (expect ~2.0)")

## 14. Key Takeaways

1. **`torch.utils.cpp_extension`** provides `load()` (JIT) and `CppExtension`/`CUDAExtension` (setuptools)
2. **`torch/extension.h`** is the single include for all PyTorch C++ functionality
3. **`AT_DISPATCH_*` macros** generate dtype-generic code without manual templates
4. **CUDA extensions** split into `.cpp` (host, gcc) and `.cu` (device, nvcc)
5. **Autograd** works via `torch::autograd::Function` in C++ or `torch.library` in Python
6. **`TORCH_CHECK`** gives clean error messages that become Python `RuntimeError`
7. **Always call `.contiguous()`** before `data_ptr<T>()`
8. **Triton is often simpler** than CUDA C++ for standard GPU patterns
9. **Register via `torch.library`** for `torch.compile` compatibility

In [ ]:
print("""
Decision tree:

  Can existing PyTorch ops do it?  ──Yes──► Stop
          │ No
  Can torch.compile fuse it?       ──Yes──► Stop
          │ No
  Is it a GPU kernel?
     ├── Yes → Try Triton first (simpler)
     │         Need low-level control? → CUDA C++ extension
     └── No  → C++ extension with OpenMP

  Need torch.compile support?
     └── Yes → Register via torch.library
""")

print("This concludes Module 36: Custom C++ Extensions.")
print("See the module README for the complete API reference.")